In [ ]:
import os
import sys
import math
import pickle
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import lr_scheduler

from torchvision import transforms, models, datasets
from torchvision.datasets import ImageFolder
from torchvision.ops import StochasticDepth

import time
import copy

from PIL import Image

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(300),
        transforms.RandomCrop(224),       
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomRotation(degrees=10),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        transforms.ColorJitter(
            brightness=0.1,
            contrast=0.1,
            saturation=0.05,
            hue=0.01
        ),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ]),

    'val': transforms.Compose([
        transforms.Resize(300),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ]),
}


In [ ]:
trainset = datasets.ImageFolder(r'/Users/ductruong/Documents/GitHub/WoodTN/data',transform=data_transforms['train'])
trainloader = torch.utils.data.DataLoader(trainset, batch_size=12,
                                          shuffle=True, num_workers=4, drop_last=False)

testset = datasets.ImageFolder(r'/Users/ductruong/Documents/GitHub/WoodTN/datal',transform=data_transforms['val'])
testloader = torch.utils.data.DataLoader(testset, batch_size=6,
                                         shuffle=False, num_workers=4, drop_last=False)

In [ ]:
dataset_sizes = {"train": len(trainset) ,
                "val": len(testset),
               }
data_loaders = {"train": trainloader,
                "val" : testloader                    
               }

In [ ]:
dataset_sizes

In [ ]:
trainset.classes

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = models.efficientnet_v2_s(pretrained=True)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features=1280, out_features=len(trainset.classes))
)
for param in model.features.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True
model = model.to(device, memory_format=torch.channels_last)

In [ ]:
def mixup_data(x, y, alpha=0.4):
    '''MixUp'''
    lam = torch.distributions.Beta(alpha, alpha).sample().item() if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def cutmix_data(x, y, alpha=1.0):
    '''CutMix'''
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    batch_size, _, H, W = x.size()
    index = torch.randperm(batch_size).to(x.device)
    y_a, y_b = y, y[index]

    rx = torch.randint(W, (1,)).item()
    ry = torch.randint(H, (1,)).item()
    rw = int(W * torch.sqrt(1 - lam).item())
    rh = int(H * torch.sqrt(1 - lam).item())
    x1 = max(rx - rw // 2, 0)
    y1 = max(ry - rh // 2, 0)
    x2 = min(rx + rw // 2, W)
    y2 = min(ry + rh // 2, H)

    x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    return x, y_a, y_b, lam

def train_model(model, criterion, optimizer, scheduler, data_loaders, dataset_sizes, device,
                num_epochs, patience=5, use_mixup=True, use_cutmix=True,
                mixup_alpha=0.4, cutmix_alpha=1.0, mix_prob=0.3):
    
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    best_acc = 0.0
    early_stop_counter = 0

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()

            running_loss = 0.0
            running_corrects = 0
            loop = tqdm(data_loaders[phase], desc=f"{phase} [{epoch+1}/{num_epochs}]")

            for inputs, labels in loop:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    # Augmentation trong TRAIN
                    if phase == 'train':
                        targets_a, targets_b, lam = labels, labels, 1.0
                        if torch.rand(1).item() < mix_prob:
                            
                            if use_mixup and use_cutmix:

                                if torch.rand(1).item() < 0.5:
                                    inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, alpha=mixup_alpha)
                                else:
                                    inputs, targets_a, targets_b, lam = cutmix_data(inputs, labels, alpha=cutmix_alpha)

                            elif use_mixup:
                                inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, alpha=mixup_alpha)
                                
                            elif use_cutmix:
                                inputs, targets_a, targets_b, lam = cutmix_data(inputs, labels, alpha=cutmix_alpha)


                        outputs = model(inputs)
                        
                        loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
                        _, preds = torch.max(outputs, 1)
                        acc_targets = targets_a
                    
                    else: # Pha 'val'
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)
                        _, preds = torch.max(outputs, 1)
                        acc_targets = labels

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == acc_targets.data)

            if phase == 'train' and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # === EARLY STOPPING ===
            if phase == 'val':
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    early_stop_counter = 0

                    os.makedirs("save_model", exist_ok=True)
                    filename_model = f"save_model/Go_v1_epoch_{epoch}.pth"
                    torch.save(best_model_wts, filename_model)
                    print(f'✅ Model saved: {filename_model}')
                else:
                    early_stop_counter += 1
                    print(f'⏳ No improvement. Early stop counter: {early_stop_counter}/{patience}')

        if early_stop_counter >= patience:
            print(f'⛔ Early stopping at epoch {epoch}')
            break

    time_elapsed = time.time() - since
    print(f'\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model

In [ ]:
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer_ft = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
exp_lr_scheduler = lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=10)

In [ ]:
model_ft = train_model(model, criterion, optimizer_ft, exp_lr_scheduler,
                      data_loaders, dataset_sizes, device, num_epochs=10, patience=3, use_mixup=True,
                      mixup_alpha=0.4, cutmix_alpha=1.0, mix_prob=0.3)

In [ ]:
# Thay X bằng số epoch tốt nhất bạn thấy trong log hoặc file đã lưu
model_ft.load_state_dict(torch.load("save_model/Go_v1_epoch_X.pth"))

In [ ]:
for param in model_ft.parameters():
    param.requires_grad = True

In [ ]:
optimizer_ft = optim.AdamW(model_ft.parameters(), lr=1e-5, weight_decay=1e-4)
exp_lr_scheduler = lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=30)

In [ ]:
model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler,
                      data_loaders, dataset_sizes, device, num_epochs=30, patience=7, use_mixup=True,
                      mixup_alpha=0.4, cutmix_alpha=1.0, mix_prob=0.3)